# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
import pandas as pd
import numpy as np

In [27]:
file1 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv")
file2 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv")
file3 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv")

for f in (file1, file2, file3):
    f.rename(columns={"State": "ST", "Gender": "GENDER"}, inplace=True)

In [28]:
df = pd.concat([file1, file2, file3], ignore_index=True)
df.head()

,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


In [29]:
df.columns

Index(['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value',
       'Income', 'Monthly Premium Auto', 'Number of Open Complaints',
       'Policy Type', 'Vehicle Class', 'Total Claim Amount'],
      dtype='object')

In [30]:
#uniformizar
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [31]:
df = df.rename(columns={
    "customer": "customer_id",
    "st": "state",
})

In [32]:
#removing columns 
df = df.drop(columns=["unnamed:_0"], errors="ignore")

In [33]:
#to show missing values
df.isna().sum()

customer_id                  2937
state                        2937
gender                       3059
education                    2937
customer_lifetime_value      2944
income                       2937
monthly_premium_auto         2937
number_of_open_complaints    2937
policy_type                  2937
vehicle_class                2937
total_claim_amount           2937
dtype: int64

In [34]:
#for numerical variables : fill Nan with 0 
df = df.fillna({col: 0 for col in df.select_dtypes(include="number").columns})

In [35]:
#for categorical variables : filling null text with unknown
df = df.fillna({col: "Unknown" for col in df.select_dtypes(include="object").columns})

In [36]:
#correcting data types
df["customer_id"] = df["customer_id"].astype(str)
df["income"] = df["income"].astype(float)

In [37]:
# identifiying column with text
cat_cols = df.select_dtypes(include="object").columns

# cleaning text
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()


In [39]:
# standardize gender values
df["gender"] = df["gender"].replace({
    "femal": "f",
    "female": "f",
    "male": "m",
})

# standardize state values
df["state"] = df["state"].replace({
    "cali": "california",
    "az": "arizona",
    "wa": "washington",
})

# standardize education spelling
df["education"] = df["education"].replace({"bachelors": "bachelor"})

# group luxury vehicle classes together
df["vehicle_class"] = df["vehicle_class"].replace({
    "sports car": "luxury",
    "luxury suv": "luxury",
    "luxury car": "luxury",
})

In [40]:
# customer_lifetime_value comes in as a percentage string
df["customer_lifetime_value"] = df["customer_lifetime_value"].astype(str).str.replace("%", "", regex=False)
df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"], errors="coerce").fillna(0)

In [41]:
# number_of_open_complaints
# number is the actual complaint count
def clean_complaints(x):
    x = str(x)
    if "/" in x:
        return int(x.split("/")[1])
    try:
        return int(float(x))
    except ValueError:
        return 0

df["number_of_open_complaints"] = df["number_of_open_complaints"].apply(clean_complaints)

In [42]:
# drop exact duplicate rows produced by combining the 3 files
df = df.drop_duplicates().reset_index(drop=True)
df.shape

(9135, 11)

In [43]:
df.isna().sum()

customer_id                  0
state                        0
gender                       0
education                    0
customer_lifetime_value      0
income                       0
monthly_premium_auto         0
number_of_open_complaints    0
policy_type                  0
vehicle_class                0
total_claim_amount           0
dtype: int64

In [44]:
df.head()

,customer_id,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,rb50392,washington,unknown,master,0.00,0.0,1000.0,0,personal auto,four-door car,2.704934
1,qz44356,arizona,f,bachelor,697953.59,0.0,94.0,0,personal auto,four-door car,1131.464935
2,ai49188,nevada,f,bachelor,1288743.17,48767.0,108.0,0,personal auto,two-door car,566.472247
3,ww63253,california,m,bachelor,764586.18,0.0,106.0,0,corporate auto,suv,529.881344
4,ga49547,washington,m,high school or below,536307.65,36357.0,68.0,0,personal auto,four-door car,17.269323


In [45]:
# saving he cleaned & combined dataset
df.to_csv("clean_combined_customer_data.csv", index=False)

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [46]:
marketing = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv")
marketing.shape

(10910, 27)

In [47]:
marketing.head()

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


In [48]:
marketing.isna().sum()

unnamed:_0                       0
customer                         0
state                            0
customer_lifetime_value          0
response                         0
coverage                         0
education                        0
effective_to_date                0
employmentstatus                 0
gender                           0
income                           0
location_code                    0
marital_status                   0
monthly_premium_auto             0
months_since_last_claim          0
months_since_policy_inception    0
number_of_open_complaints        0
number_of_policies               0
policy_type                      0
policy                           0
renew_offer_type                 0
sales_channel                    0
total_claim_amount               0
vehicle_class                    0
vehicle_size                     0
vehicle_type                     0
month                            0
dtype: int64

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [49]:
revenue_by_channel = marketing.pivot_table(
    index="sales_channel",
    values="total_claim_amount",
    aggfunc="sum"
).round(2)

revenue_by_channel = revenue_by_channel.sort_values("total_claim_amount", ascending=False)
revenue_by_channel

,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


#Insight
Agents had the most revenue (~1.81M), followed by branch, call center and web (only 0.70M), which shows that Web is underperforming relatively to the others sales channel and this could be interesting to investigate the "why".

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [50]:
clv_by_gender_education = marketing.pivot_table(
    index="gender",
    columns="education",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

clv_by_gender_education

education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05
M,7703.60,8052.46,7415.33,8149.69,8168.83


#Insight:
the average customer lifetime value shows no big difference between genders and across education levels.
we can say that based on that, the education level does not drive customer value. 